# Конспект к задаче 5

## 1. Постановка задачи

Решается двумерная задача Дирихле для уравнения Пуассона

$$
u_{xx}(x,y)+u_{yy}(x,y)=f(x,y), \qquad (x,y)\in\Omega=[x_{0},x_{1}]\times[y_{0},y_{1}].
$$

На всей границе прямоугольника задано условие Дирихле

$$
u(x,y)\big|_{\partial\Omega}=g(x,y).
$$

После аппроксимации методом конечных разностей получается линейная
система с разрежённой матрицей; в программе она решается либо прямо
(`np.linalg.solve`), либо итерационным методом Гаусса-Зейделя.

## 2. Равномерная сетка

На прямоугольнике вводится равномерная сетка с одинаковым числом
разбиений $n$ по обеим осям:

$$
x_{i}=x_{0}+i\,h_{x}, \quad i=0,1,\dots,n, \qquad h_{x}=\frac{x_{1}-x_{0}}{n},
$$

$$
y_{j}=y_{0}+j\,h_{y}, \quad j=0,1,\dots,n, \qquad h_{y}=\frac{y_{1}-y_{0}}{n}.
$$

Узлы делятся на два класса:

- **Граничные** ($i=0$, $i=n$, $j=0$ или $j=n$) — значения известны:
  $u_{i,j}=g(x_{i},y_{j})$.
- **Внутренние** ($1\le i\le n-1$, $1\le j\le n-1$) — неизвестные
  $u_{i,j}\approx u(x_{i},y_{j})$. Всего их $(n-1)^{2}$.

## 3. Пятиточечная разностная схема

Во внутренних узлах вторые производные заменяются центральными
разностями:

$$
u_{xx}(x_{i},y_{j})\approx \frac{u_{i-1,j}-2u_{i,j}+u_{i+1,j}}{h_{x}^{2}},
$$

$$
u_{yy}(x_{i},y_{j})\approx \frac{u_{i,j-1}-2u_{i,j}+u_{i,j+1}}{h_{y}^{2}}.
$$

После подстановки в уравнение Пуассона получаем разностное уравнение в
узле $(i,j)$:

$$
\frac{u_{i-1,j}-2u_{i,j}+u_{i+1,j}}{h_{x}^{2}}+\frac{u_{i,j-1}-2u_{i,j}+u_{i,j+1}}{h_{y}^{2}}=f_{i,j},
$$

где $f_{i,j}=f(x_{i},y_{j})$. Это типичный пятиточечный шаблон («крест»):
в каждое уравнение входят сам узел и его четыре ближайших соседа.

## 4. Учёт граничного условия Дирихле

Если в разностное уравнение в узле $(i,j)$ попадает сосед, лежащий на
границе, его значение $g(x_{n^{*}},y_{n^{*}})$ известно. Его вклад
переносится в правую часть, и в матрице остаются только коэффициенты
при внутренних неизвестных.

Например, если $i=1$, то сосед $(i-1,j)=(0,j)$ — граничный. Тогда член

$$
\frac{1}{h_{x}^{2}}\,u_{0,j}=\frac{1}{h_{x}^{2}}\,g(x_{0},y_{j})
$$

вычитается из правой части уравнения для узла $(1,j)$. Аналогично
поступают для остальных границ.

В коде это видно в `solver.py` в функции `build_poisson_dirichlet_system`:
для каждого соседа проверяется, граничный ли он, и если да —
коэффициент уходит в `rhs`.

## 5. Структура линейной системы

Внутренние неизвестные нумеруются построчно:

$$
\text{row}(i,j)=(j-1)\,(n-1)+(i-1), \qquad 1\le i,j\le n-1.
$$

Размер системы: $N=(n-1)^{2}$. В каждой строке матрицы $A$ ненулевыми
являются не более пяти элементов:

- на главной диагонали — $-\dfrac{2}{h_{x}^{2}}-\dfrac{2}{h_{y}^{2}}$;
- по диагоналям $\pm 1$ от главной — $\dfrac{1}{h_{x}^{2}}$ (соседи по $x$);
- по диагоналям $\pm(n-1)$ — $\dfrac{1}{h_{y}^{2}}$ (соседи по $y$).

Матрица сильно разрежённая (блочно-трёхдиагональная), но в этой лабе
для простоты хранится как полная: это удобно для прямого решателя
и для тривиальной реализации Гаусса-Зейделя через `np.flatnonzero`.

## 6. Свойства матрицы и диагональное преобладание

Для внутренней строки сумма модулей внедиагональных элементов равна

$$
\frac{2}{h_{x}^{2}}+\frac{2}{h_{y}^{2}},
$$

а модуль диагонального элемента —

$$
\left|-\frac{2}{h_{x}^{2}}-\frac{2}{h_{y}^{2}}\right|=\frac{2}{h_{x}^{2}}+\frac{2}{h_{y}^{2}}.
$$

То есть выполняется *нестрогое* диагональное преобладание

$$
|a_{ii}|\ge \sum_{j\ne i}|a_{ij}|.
$$

Для строк, соответствующих узлам, у которых хотя бы один сосед —
граничный, число ненулевых внедиагональных элементов меньше четырёх (их
вклад ушёл в правую часть). В таких строках преобладание становится
строгим:

$$
|a_{ii}|>\sum_{j\ne i}|a_{ij}|.
$$

Поскольку каждый внутренний узел в задаче с прямоугольной областью
связан хотя бы с одной граничной строкой (это видно из связности
графа узлов), матрица оказывается *иррэдуцибельно диагонально
преобладающей*. Этого достаточно, чтобы:

- матрица была невырожденной;
- метод Гаусса-Зейделя сходился для любого начального приближения.

Дополнительно матрица симметрична и отрицательно определена (это видно
из энергетического тождества для оператора Лапласа), что также
гарантирует устойчивость прямого метода.

## 7. Прямой метод (`dense`)

При значении `method = dense` система решается через `np.linalg.solve`,
которое внутри использует LU-разложение с частичным выбором ведущего
элемента. Это безусловный по сходимости метод, но память и время
зависят от размера $N=(n-1)^{2}$ как $O(N^{2})$ по памяти и $O(N^{3})$
по времени, если разрежённость матрицы не используется.

На практике для $n=160$ имеем $N\approx 2.5\cdot 10^{4}$, и полная
плотная матрица $N\times N$ занимает $\sim 5$ GB. Поэтому при больших
$n$ предпочтителен итерационный метод.

## 8. Итерационный метод Гаусса-Зейделя

При `method = seidel` система $A\mathbf{u}=\mathbf{b}$ решается
итерациями. На шаге $k+1$ для каждого уравнения $i$:

$$
u_{i}^{(k+1)}=\frac{1}{a_{ii}}\left(b_{i}-\sum_{j<i}a_{ij}\,u_{j}^{(k+1)}-\sum_{j>i}a_{ij}\,u_{j}^{(k)}\right).
$$

То есть в правой части используются уже обновлённые на этом шаге
значения для $j<i$ и старые — для $j>i$. Это отличие от метода Якоби
ускоряет сходимость и не требует дополнительной памяти на «старый»
вектор $\mathbf{u}^{(k)}$ (в коде хранится только `old = x.copy()`
ради контроля коррекции).

**Критерий останова.** Итерации прекращаются, когда максимальная
покомпонентная коррекция становится меньше заданной точности:

$$
\bigl\|\mathbf{u}^{(k+1)}-\mathbf{u}^{(k)}\bigr\|_{\infty}<\text{tol}.
$$

Дополнительно ограничено максимальное число итераций `max_iter`.

**Сходимость.** Для матрицы Пуассона диагональное преобладание плюс
симметрия и отрицательная определённость гарантируют сходимость метода
Гаусса-Зейделя из любого начального приближения. Однако скорость
сходимости пропорциональна $1/n^{2}$ — то есть при удвоении $n$ число
итераций до заданной точности растёт примерно вчетверо. Это видно по
колонке `iterations` в файле `summary.csv`.

## 9. Точность

Центральные разности во внутренних узлах имеют локальный порядок
аппроксимации $O(h_{x}^{2}+h_{y}^{2})$. При $h_{x}=h_{y}=h$ глобальная
ошибка ведёт себя как

$$
\max_{i,j}\bigl|u_{i,j}-u(x_{i},y_{j})\bigr|=O(h^{2}).
$$

При удвоении $n$ (то есть уменьшении $h$ вдвое) ошибка должна падать
примерно вчетверо. В программе это контролируется через колонки
`mean_error_ratio_prev` и `max_error_ratio_prev` в `summary.csv` —
отношение ошибки на предыдущей сетке к ошибке на текущей. Значение,
близкое к $4$, подтверждает второй порядок схемы.

## 10. Как устроена программа

Задача разбита на модули:

- `solver.py` — построение разностной системы (5-точечная схема,
  учёт граничных значений), прямой решатель `np.linalg.solve` и
  итерационный Гаусса-Зейделя.
- `input_parser.py` — чтение конфига `in.txt`, поддерживаются ключи
  `x0`, `x1`, `y0`, `y1`, `n_values`/`n`/`h_values`/`h`, `exact_n`,
  `method`, `tol`, `max_iter`.
- `csv_io.py` — запись сеточных значений в CSV: `(x, y, u)` для
  численного и точного решения, `(x, y, u_num, u_exact, delta)` для
  пофуторной сравнительной таблицы.
- `plotter.py` — построение «тепловых карт» для решений и ошибок
  через `matplotlib.imshow`.
- `main.py` — задаёт конкретные `rhs`, `exact_value`, `boundary`,
  `exact_grid` и запускает весь сценарий (решение на нескольких
  сетках, сравнение с точным, генерация графиков и summary).

## 11. Выходные файлы

После запуска `main.py` в каталоге `out/` создаются:

- `exact_grid.csv` — точное решение на частой эталонной сетке
  `(exact_n+1) x (exact_n+1)`;
- `num_n_*.csv` — численное решение для каждого значения $n$ из конфига;
- `num_vs_exact_n_*.csv` — пофуторное сравнение с точным решением
  (`x`, `y`, `u_num`, `u_exact`, `delta`);
- `comparison.png` — тепловые карты точного и численных решений
  «бок о бок»;
- `errors.png` — тепловые карты абсолютной ошибки $|u_{\text{num}}-u_{\text{exact}}|$;
- `summary.csv` — сводная таблица: $n$, $h_{x}$, $h_{y}$, число точек
  и неизвестных, число итераций (для Зейделя), нормы ошибки и невязки,
  отношения ошибок между соседними сетками.